# Extração dos loci UCE com PHYLUCE

Nesta prática usaremos o banco de matches produzido anteriormente para recuperar
as sequências UCE em FASTA.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
MATCH = ROOT / "06_uce_match"
BASE = ROOT / "07_uce_extract"
BASE.mkdir(parents=True, exist_ok=True)

CONTIGS = MATCH / "contigs"
DB = MATCH / "uce-search-results" / "probe.matches.sqlite"
MATCH_CONF = MATCH / "SRR15736591-incomplete.conf"

for p in [DB, MATCH_CONF]:
    assert p.exists(), f"Arquivo ausente: {p}. Execute primeiro o notebook 06."

## 1. Preparar PHYLUCE

In [ ]:
!mkdir -p /content/micromamba-bin
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/micromamba-bin bin/micromamba
!/content/micromamba-bin/bin/micromamba create -y -p /content/phyluce-env -c conda-forge -c bioconda phyluce=1.7.3
PHY="/content/phyluce-env/bin"

## 2. Extrair UCEs em FASTA

In [ ]:
out_fasta = BASE / "SRR15736591-incomplete.fasta"
missing = BASE / "SRR15736591-incomplete.incomplete"
logdir = BASE / "log"
logdir.mkdir(exist_ok=True)

!$PHY/phyluce_assembly_get_fastas_from_match_counts   --contigs "$CONTIGS"   --locus-db "$DB"   --match-count-output "$MATCH_CONF"   --output "$out_fasta"   --incomplete-matrix "$missing"   --log-path "$logdir"

## 3. Visualizar o FASTA

In [ ]:
!head -20 "$out_fasta"

## 4. Contar registros e comprimentos

In [ ]:
def read_fasta(path):
    records = []
    header = None
    seq = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq)))
                header = line[1:]
                seq = []
            else:
                seq.append(line)
        if header is not None:
            records.append((header, "".join(seq)))
    return records

records = read_fasta(out_fasta)
lengths = [len(s) for _,s in records]

print("Registros FASTA:", len(records))
print("Comprimento mínimo:", min(lengths) if lengths else 0)
print("Comprimento máximo:", max(lengths) if lengths else 0)
print("Comprimento médio:", round(sum(lengths)/len(lengths), 1) if lengths else 0)

## 5. Mostrar os primeiros cabeçalhos

In [ ]:
for h, s in records[:10]:
    print(h, len(s))

## Discussão

O artigo de *Hypochilus* obteve 623 loci em uma matriz de 50% de occupancy e,
após filtragem de duplicidades e potenciais problemas de homologia, manteve 550.
Nossa análise de uma única amostra e de uma fração dos reads não deve ser comparada
diretamente a esses números como se fosse uma reprodução do estudo.